# Limpieza y Transformación de Datos: US GDP vs Public Debt (1947–2020)

**Autor:** Yustin Eduardo Pérez Castro  
**Dataset:** US GDP vs Total Public Debt — datos trimestrales de la Reserva Federal (FRED) y el Departamento del Tesoro de EE.UU.  
**Fuente:** [FRED Economic Data](https://fred.stlouisfed.org/)  
**Herramientas:** Python · Pandas · NumPy

---

## Objetivo

Limpiar, estandarizar y enriquecer un dataset de series temporales macroeconómicas para dejarlo listo para análisis. El dataset cubre 295 quarters (1947 Q1 – 2020 Q3) e incluye el PIB nominal y la deuda pública total de los Estados Unidos.

### Problemas identificados en el diagnóstico inicial
1. **77 valores nulos** en la columna de deuda (26.1%) — cobertura histórica incompleta
2. **Inconsistencia de unidades** entre columnas (GDP en billions, Deuda en millions)
3. **Columna de fecha** con tipo `object` en lugar de `datetime`
4. **Columna `index` redundante** que duplica el índice de pandas
5. **Oportunidad de feature engineering** — columnas derivadas de alto valor analítico


---
## 1. Importar librerías y cargar datos


In [ ]:
import pandas as pd
import numpy as np

# Configuración de display
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', None)

# Cargar el dataset original — nunca modificamos este archivo
df_raw = pd.read_csv('data/US_GDP_vs_Debt.csv')

print(f"Dataset cargado: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
df_raw.head(10)


---
## 2. Exploración inicial (EDA de diagnóstico)

Antes de limpiar, entendemos qué tenemos. El objetivo de esta sección no es analizar los datos, sino **diagnosticar sus problemas de calidad**.


In [ ]:
# Tipos de datos y valores no nulos
df_raw.info()


In [ ]:
# Resumen estadístico
df_raw.describe()


In [ ]:
# Conteo y porcentaje de nulos por columna
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct_nulos
})
print("=== Diagnóstico de valores nulos ===")
print(resumen_nulos)


In [ ]:
# Duplicados
print(f"Filas duplicadas exactas: {df_raw.duplicated().sum()}")
print(f"Quarters duplicados: {df_raw['Quarter'].duplicated().sum()}")
print(f"\nRango temporal: {df_raw['Quarter'].iloc[0]} → {df_raw['Quarter'].iloc[-1]}")
print(f"Total de quarters en el dataset: {len(df_raw)}")
print(f"Total de quarters esperados (1947 Q1 → 2020 Q3): 295")


In [ ]:
# Identificar qué períodos tienen nulos en Deuda
deuda_nula = df_raw[df_raw['Total Public Debt ($mil)'].isnull()]
print(f"Períodos sin dato de deuda: {len(deuda_nula)}")
print(f"  Primer quarter sin dato: {deuda_nula['Quarter'].iloc[0]}")
print(f"  Último quarter sin dato (bloque histórico): {deuda_nula['Quarter'].iloc[-2]}")
print(f"  Último registro del dataset: {deuda_nula['Quarter'].iloc[-1]}")


---
## 3. Limpieza de datos

Trabajamos sobre una copia del dataset original para preservar los datos crudos intactos.


In [ ]:
# Trabajamos siempre sobre una copia
df = df_raw.copy()


### 3.1 Eliminar columna redundante

La columna `index` es simplemente una numeración secuencial (0 a 294) que duplica exactamente el índice de pandas. No agrega información analítica y puede generar confusión al trabajar con el DataFrame.


In [ ]:
df = df.drop(columns=['index'])
print("Columnas actuales:", list(df.columns))


### 3.2 Convertir la columna de fecha a tipo datetime

La columna `Quarter` está almacenada como `object` (string). Esto impide cualquier operación temporal: filtrar por año, calcular diferencias entre períodos, agrupar por década, etc.

El formato `YYYY-MM-DD` representa el primer día de cada quarter: enero (Q1), abril (Q2), julio (Q3), octubre (Q4).


In [ ]:
df['Quarter'] = pd.to_datetime(df['Quarter'])

# Verificar el cambio
print(f"Tipo anterior: object")
print(f"Tipo actual:   {df['Quarter'].dtype}")
print(f"\nEjemplos:")
print(df['Quarter'].head(6))


### 3.3 Estandarizar unidades: GDP de billions a millions

Este es el problema más sutil del dataset y el más importante de documentar.

El encabezado de ambas columnas dice `($mil)`, dando a entender que ambas están en millones. Sin embargo, al comparar los valores con fuentes externas (FRED, BEA):

- **GDP**: los valores (~243 en 1947, ~21,747 en 2020) corresponden a **miles de millones (billions) de dólares**. El PIB de EE.UU. en 2020 fue ~$21.7 trillones, que en billions = 21,747.
- **Deuda pública**: los valores (~316,000 en 1966, ~26,477,000 en 2020) corresponden a **millones de dólares**. La deuda en 2020 fue ~$26.5 trillones, que en millones = 26,477,241.

**Conclusión:** el GDP necesita multiplicarse por 1,000 para quedar expresado en millones, igual que la Deuda. De esta forma ambas columnas estarán en la misma unidad y serán directamente comparables.


In [ ]:
# Convertir GDP de billions a millions (× 1,000)
df['GDP ($mil)'] = df['Gross Domestic Product ($mil)'] * 1000
df['Debt ($mil)'] = df['Total Public Debt ($mil)']

# Eliminar columnas originales con nombre confuso
df = df.drop(columns=['Gross Domestic Product ($mil)', 'Total Public Debt ($mil)'])

# Verificar: ratio Deuda/GDP en 2020 Q2 debería ser ~135%
q2_2020 = df[df['Quarter'] == '2020-04-01'].iloc[0]
ratio = q2_2020['Debt ($mil)'] / q2_2020['GDP ($mil)'] * 100
print(f"Verificación — Deuda/GDP en 2020 Q2: {ratio:.1f}%")
print("(Valor de referencia conocido: ~135% ✓)" if 130 < ratio < 140 else "⚠ Revisar conversión")

print(f"\nGDP 2020 Q2: ${q2_2020['GDP ($mil)'] / 1e6:.2f} trillones")
print(f"Deuda 2020 Q2: ${q2_2020['Debt ($mil)'] / 1e6:.2f} trillones")


### 3.4 Tratamiento de valores nulos en Deuda

**Diagnóstico:** Los 77 valores nulos en la deuda pública se distribuyen así:
- **76 quarters** del período 1947–1965: los datos de deuda pública trimestral no se reportaban de forma sistemática en ese período.
- **1 quarter** (2020 Q3, el último registro): el dato no estaba disponible al momento de compilar el dataset.

**Decisión: no imputar.** Imputar valores de deuda en los años 1940–1960 con métodos estadísticos (media, mediana, interpolación) introduciría datos ficticios en un período históricamente significativo (posguerra, Guerra de Corea). Esto podría distorsionar cualquier análisis posterior.

**Estrategia adoptada:**
- Conservar los nulos en el dataset completo (útil para análisis solo de GDP).
- Crear un subdataset separado `df_complete` que excluya los nulos, para análisis que requieran ambas variables.


In [ ]:
# Verificar distribución de nulos
print("Distribución de nulos en Deuda:")
print(f"  1947–1965 (período histórico): {df[df['Quarter'] < '1966-01-01']['Debt ($mil)'].isnull().sum()} quarters")
print(f"  Último registro (2020 Q3):     {df[df['Quarter'] == '2020-07-01']['Debt ($mil)'].isnull().sum()} quarter")

# Dataset completo (con nulos) — para análisis solo de GDP
print(f"\ndf (dataset completo): {len(df)} filas")

# Dataset sin nulos — para análisis comparativo GDP vs Deuda
df_complete = df.dropna(subset=['Debt ($mil)']).copy()
print(f"df_complete (sin nulos en Deuda): {len(df_complete)} filas")
print(f"  Cobertura: {df_complete['Quarter'].min().date()} → {df_complete['Quarter'].max().date()}")


---
## 4. Feature engineering

Con el dataset limpio, creamos columnas derivadas que aumentan el poder analítico sin distorsionar los datos originales.


In [ ]:
# Aplicar feature engineering a ambos datasets
for dataset in [df, df_complete]:
    
    # Variables temporales
    dataset['Year']    = dataset['Quarter'].dt.year
    dataset['Q']       = dataset['Quarter'].dt.quarter.map({1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})
    dataset['Decade']  = (dataset['Year'] // 10 * 10).astype(str) + 's'

    # Crecimiento trimestral del GDP (%)
    dataset['GDP_growth_pct'] = dataset['GDP ($mil)'].pct_change() * 100

    # Flag de recesión: dos quarters consecutivos de crecimiento negativo del GDP
    gdp_neg = dataset['GDP_growth_pct'] < 0
    dataset['Recession'] = gdp_neg & gdp_neg.shift(1)

print("Columnas del dataset completo:")
print(list(df.columns))


In [ ]:
# Ratio Deuda/GDP — solo en df_complete (requiere ambas columnas sin nulos)
df_complete['Debt_to_GDP_pct'] = (df_complete['Debt ($mil)'] / df_complete['GDP ($mil)'] * 100).round(2)

# Crecimiento de la deuda trimestral (%)
df_complete['Debt_growth_pct'] = df_complete['Debt ($mil)'].pct_change() * 100

print("Columnas adicionales en df_complete:")
print([c for c in df_complete.columns if c not in df.columns])


---
## 5. Validación del output

Antes de exportar, verificamos que el dataset limpio cumple con los criterios de calidad esperados.


In [ ]:
print("=" * 50)
print("REPORTE DE VALIDACIÓN — df (dataset completo)")
print("=" * 50)

checks = {
    "Sin columna 'index' redundante":       'index' not in df.columns,
    "Columna Quarter es datetime":          str(df['Quarter'].dtype) == 'datetime64[ns]',
    "GDP en millones (valor >1M en 2020)":  df[df['Year']==2020]['GDP ($mil)'].max() > 1_000_000,
    "Sin filas duplicadas":                 df.duplicated().sum() == 0,
    "Columna Year creada correctamente":    df['Year'].between(1947, 2020).all(),
    "Columna Q con valores Q1–Q4":          set(df['Q'].unique()) == {'Q1','Q2','Q3','Q4'},
    "GDP_growth_pct calculado":             df['GDP_growth_pct'].notna().sum() > 290,
}

for descripcion, resultado in checks.items():
    estado = "✓ PASS" if resultado else "✗ FAIL"
    print(f"  {estado}  {descripcion}")

print(f"\n{'=' * 50}")
passed = sum(checks.values())
print(f"  Resultado: {passed}/{len(checks)} checks pasados")


In [ ]:
print("=" * 50)
print("REPORTE DE VALIDACIÓN — df_complete")
print("=" * 50)

checks_complete = {
    "Sin nulos en Debt ($mil)":              df_complete['Debt ($mil)'].isnull().sum() == 0,
    "Sin nulos en GDP ($mil)":               df_complete['GDP ($mil)'].isnull().sum() == 0,
    "Debt_to_GDP_pct en rango esperado":     df_complete['Debt_to_GDP_pct'].between(25, 200).all(),
    "Cobertura desde 1966":                  df_complete['Year'].min() == 1966,
    "218 filas (1966 Q1 – 2020 Q2)":        len(df_complete) == 218,
}

for descripcion, resultado in checks_complete.items():
    estado = "✓ PASS" if resultado else "✗ FAIL"
    print(f"  {estado}  {descripcion}")

passed = sum(checks_complete.values())
print(f"
  Resultado: {passed}/{len(checks_complete)} checks pasados")


---
## 6. Resumen de transformaciones


In [ ]:
print("=" * 55)
print("RESUMEN DE TRANSFORMACIONES APLICADAS")
print("=" * 55)

transformaciones = [
    ("Eliminación",       "Columna 'index' redundante eliminada"),
    ("Tipo de dato",      "Quarter: object → datetime64"),
    ("Unidades",          "GDP: billions → millions (×1,000)"),
    ("Documentación",     "77 nulos en Deuda: decisión de no imputar"),
    ("Feature eng.",      "Year, Q, Decade extraídos de la fecha"),
    ("Feature eng.",      "GDP_growth_pct: crecimiento trimestral (%)"),
    ("Feature eng.",      "Recession: flag de dos quarters negativos"),
    ("Feature eng.",      "Debt_to_GDP_pct: ratio macroeconómico clave"),
    ("Feature eng.",      "Debt_growth_pct: crecimiento trimestral deuda"),
    ("Segmentación",      "df_complete: 218 filas con ambas variables completas"),
]

for tipo, descripcion in transformaciones:
    print(f"  [{tipo:<14}] {descripcion}")

print(f"\n  Dataset original:  {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print(f"  df (completo):     {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"  df_complete:       {df_complete.shape[0]} filas × {df_complete.shape[1]} columnas")


In [ ]:
# Vista final de ambos datasets
print("=== df — últimas 5 filas ===")
print(df.tail(5).to_string(index=False))

print("\n=== df_complete — últimas 5 filas ===")
print(df_complete[['Quarter','GDP ($mil)','Debt ($mil)','Debt_to_GDP_pct','Q','Decade','Recession']].tail(5).to_string(index=False))


---
## 7. Exportar datasets limpios


In [ ]:
import os
os.makedirs('data', exist_ok=True)

# Dataset completo (todos los quarters, con NaN en Deuda para 1947-1965)
df.to_csv('data/gdp_debt_clean.csv', index=False)

# Dataset sin nulos — listo para análisis comparativo
df_complete.to_csv('data/gdp_debt_complete.csv', index=False)

print("Archivos exportados:")
print("  data/gdp_debt_clean.csv    — dataset completo (295 filas)")
print("  data/gdp_debt_complete.csv — sin nulos, listo para análisis (218 filas)")


---
## Conclusiones

Este notebook demostró cómo abordar la limpieza de un dataset de series temporales macroeconómicas con problemas reales:

### Hallazgos principales
- **El problema de unidades** (GDP en billions vs Deuda en millions) es un error de documentación del dataset original, no de los datos. Requirió validación cruzada con fuentes externas para detectarlo.
- **Los 77 nulos** en la deuda no son errores — reflejan la ausencia histórica de reportes trimestrales sistemáticos antes de 1966. Imputarlos habría introducido datos ficticios en un período históricamente significativo.
- **El ratio Deuda/GDP** pasó de ~40% en 1966 a ~135% en 2020 Q2, con el salto más abrupto durante la crisis del COVID-19 (+28 puntos porcentuales en un solo quarter).

### Próximos pasos
- `02_analisis.ipynb` — Análisis exploratorio: visualización de series temporales, correlaciones, análisis por décadas y detección de recesiones.
- `03_visualizacion.ipynb` — Dashboard con las métricas clave del dataset.
